## Apple Watch Real-Time Data — Preprocessing
Reads all `*_motion.txt` files from `data/raw_data/User_study_watch/`,
resamples to 50 Hz, and outputs a schema-conformant parquet identical
to every other Individual Dataloader.

In [ ]:
# =========================
# CELL 1 — Setup & contracts
# =========================
from pathlib import Path
import sys, re
import numpy as np
import pandas as pd

ROOT = Path("/home/aidan/IMU_LM_Data")
sys.path.insert(0, str(ROOT))

from UTILS.helpers import (
    _canon, load_contracts, to_continuous_stream,
    run_qa_checks, check_sample_integrity,
    _split_on_gaps_seconds,
)

C          = load_contracts(ROOT)
SCHEMA     = C["SCHEMA"]
RAW2ID     = C["RAW2ID"]
ID2NAME    = C["ID2NAME"]
UNKNOWN_ID = C["UNKNOWN_ID"]
TARGET_HZ  = C["TARGET_HZ"]
CLEANED    = C["CLEANED"]

RAW_DIR      = ROOT / "data" / "raw_data" / "User_study_watch"
DATASET_NAME = "apple_watch_study"
GAP_CUTOFF_S = 0.5          # split into new segment if gap > 500 ms
G_MS2        = 9.80665      # standard gravity

# ---- Apple Watch CoreMotion 21-column layout (space-separated) ----
# 0        unix_ts              (s)
# 1-3      userAcceleration     (g, gravity-removed)
# 4-6      gravity              (normalised direction)
# 7-9      rotationRate         (rad/s)
# 10-12    magneticField        (µT)
# 13-15    attitude euler       (roll, pitch, yaw — rad)
# 16-19    attitude quaternion   (x, y, z, w)
# 20       elapsed_time         (s, monotonic clock)
WATCH_COLS = [
    "unix_ts",
    "user_acc_x", "user_acc_y", "user_acc_z",
    "gravity_x",  "gravity_y",  "gravity_z",
    "gyro_x",     "gyro_y",     "gyro_z",
    "mag_x",      "mag_y",      "mag_z",
    "euler_roll", "euler_pitch", "euler_yaw",
    "quat_x",    "quat_y",     "quat_z",    "quat_w",
    "elapsed_s",
]

# filename pattern: {Subject}_{YY-MM-DD}_{HH-MM-SS}_motion.txt
FNAME_RE = re.compile(
    r"^(.+?)_(\d{2}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})_motion\.txt$"
)

motion_files = sorted(RAW_DIR.glob("*_motion.txt"))
print(f"RAW_DIR:    {RAW_DIR}")
print(f"TARGET_HZ:  {TARGET_HZ}")
print(f"Files:      {len(motion_files)}")
for f in motion_files:
    print(f"  • {f.name}")

In [ ]:
# =========================
# CELL 2 — Parse, resample to 50 Hz, build native frame
# =========================

def _resample_segment(seg_df, sensor_cols, target_hz=50):
    """Interpolate sensor columns onto a uniform 50 Hz grid."""
    t = seg_df["unix_ts"].to_numpy(np.float64)
    if t.size < 2:
        return pd.DataFrame()
    dt = 1.0 / target_hz
    grid = np.arange(t[0], t[-1] + 1e-12, dt, dtype=np.float64)
    if grid.size < 2:
        return pd.DataFrame()
    out = pd.DataFrame({"unix_ts": grid})
    for c in sensor_cols:
        y = seg_df[c].to_numpy(np.float64)
        m = np.isfinite(t) & np.isfinite(y)
        if m.sum() >= 2:
            out[c] = np.interp(grid, t[m], y[m]).astype(np.float32)
        else:
            out[c] = np.nan
    return out.dropna(subset=sensor_cols).reset_index(drop=True)


SENSOR_COLS = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
all_native = []

for fpath in motion_files:
    # ---- parse filename ----
    m = FNAME_RE.match(fpath.name)
    if not m:
        print(f"[SKIP] unrecognised filename: {fpath.name}")
        continue
    subject, session_tag = m.group(1), m.group(2)

    # ---- read 21-column space-delimited file ----
    raw = pd.read_csv(
        fpath, sep=r"\s+", header=None, names=WATCH_COLS,
        dtype=np.float64, on_bad_lines="skip",
    )
    if raw.empty or len(raw) < 2:
        print(f"[SKIP] empty/short: {fpath.name}")
        continue

    # ---- derive total acceleration (g-included) in m/s² ----
    raw["acc_x"] = (raw["gravity_x"] + raw["user_acc_x"]) * G_MS2
    raw["acc_y"] = (raw["gravity_y"] + raw["user_acc_y"]) * G_MS2
    raw["acc_z"] = (raw["gravity_z"] + raw["user_acc_z"]) * G_MS2
    # gyro_x/y/z already in rad/s — keep as-is

    raw = raw.sort_values("unix_ts").dropna(
        subset=["unix_ts"] + SENSOR_COLS
    ).reset_index(drop=True)

    est_hz = 1.0 / np.median(np.diff(raw["unix_ts"].values))
    print(f"{fpath.name}  |  {len(raw):,} rows  |  raw Hz ≈ {est_hz:.1f}")

    # ---- split on time-gaps, resample each segment to 50 Hz ----
    segments = _split_on_gaps_seconds(raw, "unix_ts", GAP_CUTOFF_S)
    for seg_i, seg in enumerate(segments):
        rs = _resample_segment(seg, SENSOR_COLS, TARGET_HZ)
        if rs.empty:
            continue
        rs["dataset"]                = DATASET_NAME
        rs["subject_id"]             = subject
        rs["session_id"]             = f"{session_tag}_seg{seg_i:03d}"
        rs["timestamp_ns"]           = (rs["unix_ts"] * 1e9).round().astype("int64")
        rs["dataset_activity_id"]    = np.int16(UNKNOWN_ID)
        rs["dataset_activity_label"] = "unlabeled"
        all_native.append(rs)

if not all_native:
    raise SystemExit("No data produced — check RAW_DIR contents.")

native = pd.concat(all_native, ignore_index=True)[[
    "dataset", "subject_id", "session_id", "timestamp_ns",
    "acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z",
    "dataset_activity_id", "dataset_activity_label",
]].copy()

native["dataset"]                = native["dataset"].astype("string")
native["subject_id"]             = native["subject_id"].astype("string")
native["session_id"]             = native["session_id"].astype("string")
native["dataset_activity_label"] = native["dataset_activity_label"].astype("string")

print(f"\nNative frame: {len(native):,} rows  |  "
      f"{native['subject_id'].nunique()} subject(s)  |  "
      f"{native['session_id'].nunique()} session(s)")
native.head(3)

In [ ]:
# =========================
# CELL 3 — Schema conversion, QA, save
# =========================

df_final = to_continuous_stream(native, SCHEMA, RAW2ID, ID2NAME, UNKNOWN_ID)
print(f"Schema-aligned rows: {len(df_final):,}")
print(f"Columns: {list(df_final.columns)}\n")

run_qa_checks(df_final, SCHEMA, UNKNOWN_ID)
check_sample_integrity(df_final, SCHEMA)

# ---- Save ----
out_path = CLEANED / f"{DATASET_NAME}_clean_data.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
df_final.to_parquet(out_path, index=False)
print(f"\n✓ Saved → {out_path}  ({out_path.stat().st_size / 1e6:.2f} MB)")